#Laboratorio Visualización interactiva con Python sesion 25

**¿De qué manera la expansión comercial y el volumen facturado por Adventure Works entre 2005 y 2008 impactaron la rentabilidad real del negocio en sus distintas líneas y mercados, y en qué medida las categorías y regiones de mayor venta encubren pérdidas o márgenes deficitarios que comprometen la utilidad bruta corporativa?**

##Librerias

In [15]:
import re
import unicodedata
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import glob

# 1. Configuración de Identidad Visual
VERDE_CICLISMO   = '#1B5E20'  # Verde Ruta / Naturaleza principal
VERDE_SUAVE      = '#34D399'  # Verde Menta / Bianchi acento
NARANJA_RACING   = '#F97316'  # Naranja Alta Visibilidad / Energía
GRIS_CARBON      = '#1E293B'  # Fibra de carbono / Textos principales
GRIS_ALUMINIO    = '#64748B'  # Aleación metálica / Ejes y contexto
GRIS_PISTA       = '#F1F5F9'  # Asfalto claro / Cuadrículas
BLANCO           = '#FFFFFF'

##Preparación de los datos

In [16]:
def normalizar(texto):
    """Estandariza cadenas de texto para búsqueda de archivos y columnas."""
    texto = unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode().lower().strip()
    return re.sub(r'[^a-z0-9]+', '_', texto).strip('_')

def buscar_archivo(base, extensiones):
    """Busca un archivo en las rutas habituales de trabajo según su nombre y extensiones."""
    archivos = [p for p in glob.glob('/content/*') + glob.glob('/mnt/data/*') + glob.glob('./*') if os.path.isfile(p)]
    candidatos = []
    for p in archivos:
        nombre = normalizar(os.path.splitext(os.path.basename(p))[0])
        ext = os.path.splitext(p)[1].lower()
        if ext in extensiones and (nombre == base or re.fullmatch(re.escape(base) + r'_?[0-9]+', nombre)):
            candidatos.append(p)
    candidatos.sort(key=lambda p: (normalizar(os.path.splitext(os.path.basename(p))[0]) != base, len(p)))
    return candidatos[0] if candidatos else None

# Búsqueda automática del archivo
nombre_base = normalizar('Sábana de datos')
ruta_excel = buscar_archivo(nombre_base, ('.xlsx', '.xls'))

# Si está en Google Colab y no se ha subido aún, solicitar la carga
if ruta_excel is None:
    try:
        from google.colab import files
        print("Seleccione y suba el archivo 'Sábana de datos.xlsx':")
        subidos = files.upload()
        ruta_excel = buscar_archivo(nombre_base, ('.xlsx', '.xls'))
    except ImportError:
        pass

if ruta_excel is None:
    raise FileNotFoundError("No se encontró el archivo 'Sábana de datos.xlsx'. Verifique la carpeta de trabajo o súbalo al entorno.")

print(f"Archivo localizado: {ruta_excel}")

# Carga de la hoja solicitada en la guía
df = pd.read_excel(ruta_excel, sheet_name='Ventas AdventureWorks')
print(f"Dataset cargado con éxito: {df.shape[0]:,} filas y {df.shape[1]} columnas.")

Archivo localizado: ./Sábana de datos.xlsx
Dataset cargado con éxito: 60,852 filas y 39 columnas.


In [19]:
# Imputaciones justificadas de calidad de datos
df['Categoria'] = df['Categoria'].fillna('Accesorio')
df['País'] = df['País'].fillna('Francia')

# Normalización de fechas y cálculos
df['Fecha Venta'] = pd.to_datetime(df['Fecha Venta'], errors='coerce')
df['Mes'] = df['Fecha Venta'].dt.to_period('M').dt.to_timestamp()
df['Utilidad Bruta'] = df['Venta Neta'] - df['Costo Total']

# Totales globales para evitar errores de referencia
total_ventas = df['Venta Neta'].sum()
total_costo = df['Costo Total'].sum()
utilidad_total = total_ventas - total_costo
margen_total = (utilidad_total / total_ventas) * 100

categorias = ['Todas', 'Bicicleta', 'Componente', 'Prenda', 'Accesorio']

##EDA datos

In [14]:
# ------------------------------------------------------------------------------
# 2. DIAGNÓSTICO DE CALIDAD DE DATOS (Nulos, Duplicados y Tipos)
# ------------------------------------------------------------------------------
print("=== 2.1 CONTEO DE VALORES NULOS POR VARIABLE ===")
nulos = df.isnull().sum()
nulos_tabla = pd.DataFrame({
    'Valores_Faltantes': nulos[nulos > 0],
    'Porcentaje_%': (nulos[nulos > 0] / len(df) * 100).round(2)
}).sort_values(by='Valores_Faltantes', ascending=False)
print(nulos_tabla)

print("\n=== 2.2 VERIFICACIÓN DE UNICIDAD ===")
print(f"Filas duplicadas exactas: {df.duplicated().sum():,}")
print(f"Transacciones únicas (Consecutivo): {df['Consecutivo Transaccion'].nunique():,}")
print(f"Facturas únicas: {df['Factura'].nunique():,}")

# ------------------------------------------------------------------------------
# 3. PREPROCESAMIENTO Y LIMPIEZA JUSTIFICADA
# ------------------------------------------------------------------------------
# A) Imputación de 'Categoria' (Fila con Cod Categoria = 4 y Casco -> Accesorio)
fila_nan_cat = df['Categoria'].isna()
if fila_nan_cat.any():
    df.loc[fila_nan_cat, 'Categoria'] = 'Accesorio'

# B) Imputación de 'Costo Total' faltante mediante (Cantidad * Costo Unitario)
fila_nan_costo = df['Costo Total'].isna()
if fila_nan_costo.any():
    df.loc[fila_nan_costo, 'Costo Total'] = (
        df.loc[fila_nan_costo, 'Cantidad'] * df.loc[fila_nan_costo, 'Costo Unitario']
    )

# C) Imputación de 'Venta Total' faltante si aplica
fila_nan_venta = df['Venta Total'].isna()
if fila_nan_venta.any():
    df.loc[fila_nan_venta, 'Venta Total'] = df.loc[fila_nan_venta, 'Venta Neta']

# D) Imputación fundamentada de 'País':
# Todos los registros con País = NaN corresponden al Territorio 7 (Francia / Europa)
fila_nan_pais = df['País'].isna()
if fila_nan_pais.any():
    df.loc[fila_nan_pais, 'País'] = 'France'
    df.loc[fila_nan_pais, 'Region'] = 'France'

# E) Estandarización de Fechas y variables temporales
df['Fecha Venta'] = pd.to_datetime(df['Fecha Venta'], errors='coerce')
df['Año'] = df['Fecha Venta'].dt.year
df['Mes'] = df['Fecha Venta'].dt.month
df['Periodo_YM'] = df['Fecha Venta'].dt.to_period('M').astype(str)

# F) Creación de Variables Analíticas Financieras
df['Utilidad Bruta'] = df['Venta Neta'] - df['Costo Total']
df['Margen_Rentabilidad_%'] = np.where(
    df['Venta Neta'] != 0,
    (df['Utilidad Bruta'] / df['Venta Neta']) * 100,
    0.0
)

print("\n-> Preprocesamiento completado exitosamente sin pérdida indiscriminada de filas.")

# ------------------------------------------------------------------------------
# 4. EXPLORACIÓN ESTADÍSTICA (EDA) Y DESBALANCES ESTRUCTURALES
# ------------------------------------------------------------------------------
total_ventas = df['Venta Neta'].sum()
total_costos = df['Costo Total'].sum()
total_utilidad = df['Utilidad Bruta'].sum()
margen_promedio = (total_utilidad / total_ventas) * 100

print("\n" + "="*70)
print(f"MÉTRICAS TOTALES DE ADVENTURE WORKS (2005 - 2008)")
print(f"Total Venta Neta:       ${total_ventas:,.2f} USD")
print(f"Total Costo de Ventas:  ${total_costos:,.2f} USD")
print(f"Total Utilidad Bruta:   ${total_utilidad:,.2f} USD")
print(f"Margen Bruto Global:    {margen_promedio:.2f}%")
print(f"Total Unidades:         {df['Cantidad'].sum():,} unidades")
print("="*70)

# 4.1 Desbalance por Categoría de Producto
print("\n--- RESUMEN 1: DESBALANCE ESTRUCTURAL POR CATEGORÍA ---")
eda_cat = df.groupby('Categoria').agg(
    Transacciones=('Venta Neta', 'count'),
    Unidades=('Cantidad', 'sum'),
    Venta_Neta=('Venta Neta', 'sum'),
    Costo_Total=('Costo Total', 'sum'),
    Utilidad_Bruta=('Utilidad Bruta', 'sum')
)
eda_cat['Part_Ventas_%'] = (eda_cat['Venta_Neta'] / total_ventas) * 100
eda_cat['Margen_Real_%'] = (eda_cat['Utilidad_Bruta'] / eda_cat['Venta_Neta']) * 100
eda_cat = eda_cat.sort_values(by='Venta_Neta', ascending=False)
print(eda_cat[['Part_Ventas_%', 'Venta_Neta', 'Utilidad_Bruta', 'Margen_Real_%']].to_string(
    formatters={
        'Part_Ventas_%': '{:,.2f}%'.format,
        'Venta_Neta': '${:,.2f}'.format,
        'Utilidad_Bruta': '${:,.2f}'.format,
        'Margen_Real_%': '{:,.2f}%'.format
    }
))

# 4.2 Desbalance Geográfico por País
print("\n--- RESUMEN 2: RENTABILIDAD POR MERCADO / PAÍS ---")
eda_pais = df.groupby('País').agg(
    Transacciones=('Venta Neta', 'count'),
    Venta_Neta=('Venta Neta', 'sum'),
    Utilidad_Bruta=('Utilidad Bruta', 'sum')
)
eda_pais['Part_Ventas_%'] = (eda_pais['Venta_Neta'] / total_ventas) * 100
eda_pais['Margen_%'] = (eda_pais['Utilidad_Bruta'] / eda_pais['Venta_Neta']) * 100
eda_pais = eda_pais.sort_values(by='Venta_Neta', ascending=False)
print(eda_pais[['Part_Ventas_%', 'Venta_Neta', 'Utilidad_Bruta', 'Margen_%']].to_string(
    formatters={
        'Part_Ventas_%': '{:,.2f}%'.format,
        'Venta_Neta': '${:,.2f}'.format,
        'Utilidad_Bruta': '${:,.2f}'.format,
        'Margen_%': '{:,.2f}%'.format
    }
))

# 4.3 Evolución Temporal Anual (2005 - 2008)
print("\n--- RESUMEN 3: EVOLUCIÓN HISTÓRICA POR AÑO ---")
eda_ano = df.groupby('Año').agg(
    Transacciones=('Venta Neta', 'count'),
    Venta_Neta=('Venta Neta', 'sum'),
    Utilidad_Bruta=('Utilidad Bruta', 'sum')
)
eda_ano['Margen_%'] = (eda_ano['Utilidad_Bruta'] / eda_ano['Venta_Neta']) * 100
print(eda_ano.to_string(
    formatters={
        'Venta_Neta': '${:,.2f}'.format,
        'Utilidad_Bruta': '${:,.2f}'.format,
        'Margen_%': '{:,.2f}%'.format
    }
))

=== 2.1 CONTEO DE VALORES NULOS POR VARIABLE ===
                  Valores_Faltantes  Porcentaje_%
Clase                         18467         30.35
Color                          5226          8.59
Region                         3530          5.80
Modelo                          861          1.41
Tipo Negocio                    354          0.58
Estado Actual                     2          0.00
Codigo Geografia                  2          0.00
Ciudad                            1          0.00
Costo Total                       1          0.00
Vendedor                          1          0.00
Venta Total                       1          0.00
Utilidad Bruta                    1          0.00

=== 2.2 VERIFICACIÓN DE UNICIDAD ===
Filas duplicadas exactas: 0
Transacciones únicas (Consecutivo): 60,852
Facturas únicas: 3,796

-> Preprocesamiento completado exitosamente sin pérdida indiscriminada de filas.

MÉTRICAS TOTALES DE ADVENTURE WORKS (2005 - 2008)
Total Venta Neta:       $80,450,470.

##Subplots

In [20]:
# 3. Creación del Lienzo de Subplots
fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{'type': 'xy'}, {'type': 'xy'}],
        [{'type': 'xy', 'colspan': 2}, None],
    ],
    subplot_titles=(
        'Evolución Mensual de Ventas Netas',
        'Venta Neta vs. Utilidad Bruta por Categoría',
        'Distribución de Ventas por País / Mercado',
    ),
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

traces_per_cat = 4  # 1 temporal, 2 barras categoria, 1 columnas pais
total_traces = len(categorias) * traces_per_cat

##Traza

In [21]:
for i, cat in enumerate(categorias):
    sub_df = df if cat == 'Todas' else df[df['Categoria'] == cat]

    # Datos Gráfico 1: Temporal
    mensual = sub_df.groupby('Mes')['Venta Neta'].sum().reset_index().sort_values('Mes')

    # Datos Gráfico 2: Categorías
    cat_agg = sub_df.groupby('Categoria')[['Venta Neta', 'Utilidad Bruta']].sum().reset_index().sort_values('Venta Neta', ascending=True)

    # Datos Gráfico 3: Países
    pais_agg = sub_df.groupby('País')['Venta Neta'].sum().reset_index().sort_values('Venta Neta', ascending=False)

    is_visible = (i == 0)

    # 1. Gráfico de Líneas (Evolución Mensual) - Verde Ruta con área sombreada
    fig.add_trace(
        go.Scatter(
            x=mensual['Mes'],
            y=mensual['Venta Neta'],
            mode='lines+markers',
            name=f'Venta ({cat})',
            line=dict(color=VERDE_CICLISMO, width=2.8),
            marker=dict(size=6, color=VERDE_CICLISMO),
            fill='tozeroy',
            fillcolor='rgba(27, 94, 32, 0.08)',
            visible=is_visible,
            hovertemplate='<b>Mes:</b> %{x|%b %Y}<br><b>Venta Neta:</b> $%{y:,.0f} USD<extra></extra>'
        ),
        row=1, col=1
    )

    # 2. Barras Horizontales - Ventas Netas (Verde Ciclismo)
    fig.add_trace(
        go.Bar(
            y=cat_agg['Categoria'],
            x=cat_agg['Venta Neta'],
            orientation='h',
            name='Venta Neta',
            marker_color=VERDE_CICLISMO,
            visible=is_visible,
            hovertemplate='<b>%{y}</b><br>Venta Neta: $%{x:,.0f} USD<extra></extra>'
        ),
        row=1, col=2
    )

    # 3. Barras Horizontales - Utilidad Bruta (Naranja Racing para destacar rentabilidad)
    fig.add_trace(
        go.Bar(
            y=cat_agg['Categoria'],
            x=cat_agg['Utilidad Bruta'],
            orientation='h',
            name='Utilidad Bruta',
            marker_color=NARANJA_RACING,
            visible=is_visible,
            hovertemplate='<b>%{y}</b><br>Utilidad Bruta: $%{x:,.0f} USD<extra></extra>'
        ),
        row=1, col=2
    )

    # 4. Columnas por País (Verde Esmeralda con borde sutil)
    fig.add_trace(
        go.Bar(
            x=pais_agg['País'],
            y=pais_agg['Venta Neta'],
            name='Ventas por País',
            marker=dict(
                color=VERDE_CICLISMO,
                line=dict(color=GRIS_CARBON, width=1)
            ),
            visible=is_visible,
            text=pais_agg['Venta Neta'],
            texttemplate='$%{text:,.2s}',
            textposition='outside',
            hovertemplate='<b>País:</b> %{x}<br><b>Venta Neta:</b> $%{y:,.0f} USD<extra></extra>'
        ),
        row=2, col=1
    )

##Menú Interactivo

In [25]:
buttons = []
for i, cat in enumerate(categorias):
    vis = [False] * total_traces
    vis[i * traces_per_cat : (i + 1) * traces_per_cat] = [True] * traces_per_cat

    sub_df = df if cat == 'Todas' else df[df['Categoria'] == cat]
    v_tot = sub_df['Venta Neta'].sum()
    u_tot = sub_df['Utilidad Bruta'].sum()
    m_pct = (u_tot / v_tot) * 100 if v_tot != 0 else 0

    title_text = (
        f"<b>Adventure Works Cycles · Monitoreo Comercial y Rentabilidad</b>"
        f"<br><span style='font-size:12px; color:{GRIS_ALUMINIO}'>"
        f"Línea Activa: <b>{cat}</b> | Facturación Neta: <b>${v_tot:,.0f} USD</b> | Margen Operativo: <b>{m_pct:.2f}%</b></span>"
    )

    buttons.append(dict(
        label=f"Filtrar: {cat}",
        method="update",
        args=[{"visible": vis}, {"title.text": title_text}]
    ))

##Formato

In [26]:
fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            # --- REUBICACIÓN EN ESQUINA SUPERIOR DERECHA ---
            x=0.98,              # Alineado al extremo derecho
            xanchor="right",     # El anclaje se expande hacia la izquierda
            y=1.14,              # A la misma altura del título sin tocarlo
            yanchor="top",
            bgcolor=BLANCO,
            bordercolor='#D9D9D9',
            borderwidth=1,
            font=dict(color=GRIS_CARBON, size=11)
        )
    ],
    title=dict(
        text=(
            "<b>Adventure Works Cycles · Monitoreo Comercial y Rentabilidad</b>"
            f"<br><span style='font-size:12px; color:{GRIS_ALUMINIO}'>"
            f"Línea Activa: <b>Todas</b> | Facturación Neta: <b>${total_ventas:,.0f} USD</b> | Margen Operativo: <b>{margen_total:.2f}%</b></span>"
        ),
        x=0.02,
        xanchor="left",
        font=dict(size=17, color=GRIS_CARBON)
    ),
    font=dict(family="Arial, sans-serif", size=11, color=GRIS_CARBON),
    paper_bgcolor=BLANCO,
    plot_bgcolor=BLANCO,
    height=840,
    # Margen superior con espacio suficiente (t=130) para título y filtro
    margin=dict(l=70, r=40, t=130, b=70),
    showlegend=False,
    hovermode='closest'
)

# Ajuste de ejes y rejillas tenues
fig.update_xaxes(showgrid=False, linecolor='#CBD5E1', row=1, col=1, title="Mes de Venta")
fig.update_yaxes(gridcolor=GRIS_PISTA, zeroline=False, row=1, col=1, title="Venta Neta (USD)")

fig.update_xaxes(showgrid=True, gridcolor=GRIS_PISTA, zeroline=True, zerolinecolor='#94A3B8', row=1, col=2, title="Monto (USD)")
fig.update_yaxes(showgrid=False, linecolor='#CBD5E1', row=1, col=2)

fig.update_xaxes(showgrid=False, linecolor='#CBD5E1', row=2, col=1, title="Mercado / País")
fig.update_yaxes(gridcolor=GRIS_PISTA, zeroline=False, row=2, col=1, title="Total Ventas Netas (USD)")

##HTML

In [28]:
from google.colab import files

# 1. Guarda el archivo en Colab
nombre_salida = 'Mini_Dashboard_AdventureWorks_Cycles.html'
fig.write_html(nombre_salida, include_plotlyjs=True, full_html=True)
print(f"✅ Archivo creado localmente: {nombre_salida}")

# 2. Dispara la descarga a tu computadora
files.download(nombre_salida)

✅ Archivo creado localmente: Mini_Dashboard_AdventureWorks_Cycles.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Hallazgos

### **Hallazgo 1: La paradoja del producto estrella (Vender más no significa ganar más)**

* **El producto principal genera pérdidas:** La categoría de **Bicicletas** representa más del **$82\%$** de todo el dinero que ingresa a la empresa (alrededor de **$\$66.3$ millones de USD**). Sin embargo, lejos de generar ganancias, arroja una **pérdida neta acumulada de casi $-\$1$ millón de USD** ($-\$990{,}700\text{ USD}$), con un margen negativo de **$-1.49\%$**.
* **Las líneas secundarias salvan el negocio:** La rentabilidad de la compañía sobrevive gracias a categorías más pequeñas como **Componentes** (con ganancias de **$+\$1.03$ millones de USD** y un margen de **$+8.75\%$**) y **Accesorios** (con un margen positivo de **$+34.28\%$**).
---

### **Hallazgo 2: Fugas de rentabilidad en la expansión internacional**

* **Dependencia de Norteamérica:** Dos países, **Estados Unidos y Canadá**, representan el **$84.5\%$** de las ventas globales de la empresa. Aunque mueven la mayor parte del mercado, sus márgenes de ganancia son bastante reducidos (apenas entre el **$+1.0\%$** y **$+1.2\%$**).
* **Pérdidas operativas en el extranjero:** Los mercados internacionales fuera de Norteamérica no están siendo sostenibles. En particular, filiales como **Australia** (margen de **$-6.82\%$**) y **Alemania** (margen de **$-5.61\%$**) presentan números rojos evidentes.